In [1]:
import deepchem as dc
import os
import pandas as pd
from sklearn.model_selection import train_test_split
os.environ['ENABLE_LOGGING'] = 'false'

Skipped loading some Pytorch utilities, missing a dependency. No module named 'torch'
No normalization for SPS. Feature removed!
No normalization for AvgIpc. Feature removed!
Skipped loading PyTorch datasets, missing a dependency. No module named 'torch'
Skipped loading some Tensorflow models, missing a dependency. No module named 'tensorflow'


This module requires PyTorch to be installed.


Skipped loading some PyTorch models, missing a dependency. No module named 'torch'
No module named 'torch'
Skipped loading modules with pytorch-geometric dependency, missing a dependency. No module named 'torch'
Skipped loading modules with pytorch-lightning dependency, missing a dependency. No module named 'torch'
Skipped loading some Jax models, missing a dependency. No module named 'jax'
Skipped loading some PyTorch models, missing a dependency. No module named 'tensorflow'


In [2]:
from deepretro.models.hallucination_trainer import HallucinationTrainer
from deepretro.models.hallucination_checker import HallucinationChecker

In [3]:
WORKSPACE_DIR = "./hallucination_workspace_ohu_neg_v3_jun16"

# STAGE 1: THE TRAINING & OPTIMIZATION PHASE (Trainer)

trainer = HallucinationTrainer(
    trainer_dir=WORKSPACE_DIR,
    model_type="xgboost",
    n_tasks=1
)


In [4]:
# Load and featurize the explicit training and testing partitions.
# CSVLoader parses the compound tuples through our internal ReactionStepFeaturizer
full_data = '/home/riya/Documents/Chiron/DeepRetro/data/data_jun23/Rishi_plus_gemini_plus_23_jun.csv'

df = pd.read_csv(full_data)

# Stratified split
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["final_label"]
)

train_csv=f"{WORKSPACE_DIR}/train.csv"
test_csv=f"{WORKSPACE_DIR}/test.csv"

# Save to CSV
train_df.to_csv(train_csv, index=False)
test_df.to_csv(test_csv, index=False)

print("Saved train.csv and test.csv")

Saved train.csv and test.csv


In [5]:

train_dataset, test_dataset = trainer.load_dataset(
    train_csv=train_csv,
    test_csv=test_csv,
    product_col="product",
    reactants_col="reactants",
    label_col="final_label"
)


Loading training data from ./hallucination_workspace_ohu_neg_v3_jun16/train.csv


[22:22:30] SMILES Parse Error: unclosed ring for input: 'O=C1[C@@]2(CC(C)C)N([C@]3([H])C(N2)=O)[C@](CC(C)C)(C(=O)Cl)N1'
[22:22:30] SMILES Parse Error: unclosed ring for input: 'O[C@H]3C'
[22:22:30] SMILES Parse Error: unclosed ring for input: 'O=C1[C@@]2(CC(C)C)N([C@]3([H])C(N2)=O)[C@](CC(C)C)(C(O)=O)N1'
[22:22:30] SMILES Parse Error: unclosed ring for input: 'O[C@@H]3C'
[22:22:30] SMILES Parse Error: unclosed ring for input: 'O=C1[C@@]2(CC(C)C)N([C@]3([H])C(N2)=O)[C@](CC(C)C)(C(=O)Cl)N1'
[22:22:30] SMILES Parse Error: unclosed ring for input: 'O[C@H]3C'
[22:22:30] WARNING: not removing hydrogen atom without neighbors
[22:22:30] WARNING: not removing hydrogen atom without neighbors
[22:22:31] SMILES Parse Error: unclosed ring for input: 'O=C1[C@@]2(CC(C)C)N([C@]3([H])C(N2)=O)[C@](CC(C)C)(C(=O)Cl)N1'
[22:22:31] SMILES Parse Error: unclosed ring for input: '[H]O[C@H]3C'
[22:22:31] SMILES Parse Error: unclosed ring for input: 'O=C1[C@@]2(CC(C)C)N([C@]3([H])C(N2)=O)[C@](CC(C)C)(C(O)=O)N1'


Loading testing data from ./hallucination_workspace_ohu_neg_v3_jun16/test.csv


[22:22:35] SMILES Parse Error: unclosed ring for input: 'O=C1[C@@]2(CC(C)C)N([C@]3([H])C(N2)=O)[C@](CC(C)C)(C(O)=O)N1'
[22:22:35] SMILES Parse Error: unclosed ring for input: 'O[C@@H](C)[C@H]3N'
[22:22:35] SMILES Parse Error: unclosed ring for input: 'O=C1[C@@]2(CC(C)C)N([C@]3([H])C(N2)=O)[C@](CC(C)C)(C(O)=O)N1'
[22:22:35] SMILES Parse Error: unclosed ring for input: 'O[C@@H]3C'
[22:22:35] SMILES Parse Error: unclosed ring for input: 'O=C1[C@@]2(CC(C)C)N([C@]3([H])C(N2)=O)[C@](CC(C)C)(C(OC)=O)N1'
[22:22:35] SMILES Parse Error: unclosed ring for input: 'O[C@@H]3C'
[22:22:35] SMILES Parse Error: unclosed ring for input: 'O=C1[C@@]2(CC(C)C)N([C@]3([H])C(O)=O)[C@](CC(C)C)(C(O[C@H]3C)=O)N1'
[22:22:35] SMILES Parse Error: unclosed ring for input: '[H]N2'
[22:22:35] SMILES Parse Error: unclosed ring for input: 'O=C1[C@@]2(CC(C)C)N([C@]3([H])C(N2)=O)[C@](CC(C)C)(C(=O)O)N1'
[22:22:35] SMILES Parse Error: unclosed ring for input: 'O[C@@H]3C'
[22:22:35] SMILES Parse Error: unclosed ring for input

In [6]:

# Execute training. We flag 'tune_params=True' and specify 'k_folds=2' to leverage 
# our custom KFoldRandomHyperparamOpt optimizer child class.
print("\nStarting automated optimization loop...")
final_model, test_performance = trainer.train_model(
    train_dataset=train_dataset,
    test_dataset=test_dataset,
    tune_params=True,
    n_trials=2,   # Kept low for execution speed in this example
    k_folds=5     # Runs cross-validation to isolate parameters and evaluate average optimal threshold
)


Starting automated optimization loop...

--- Initiating Pre-Training Parameter Tuning ---
Starting K-Fold (5) random hyperparameter tuning with 2 trials...
Calculating average optimal threshold across folds for best parameters...

--- Tuning Complete ---
Best Params: {'eta': 0.2, 'max_depth': 8, 'gamma': 0.0, 'min_child_weight': 3, 'subsample': 0.8, 'colsample_bytree': 0.7, 'reg_lambda': 10.0, 'reg_alpha': 5.0}
Best Threshold: 0.5091 (PRC-AUC Score: 0.0000)

--- Building and Training Final XGBOOST Model ---

--- Final Training Complete. Evaluating on Test Dataset ---
Evaluating dataset...
Evaluation Scores: {'ROC_AUC': 0.9341091856382375, 'PRC_AUC': 0.9237332741168237, 'threshold_accuracy': 0.8894736842105263, 'threshold_precision': 0.9054054054054054, 'threshold_recall': 0.8271604938271605}

--- Saving Model and Configuration ---
Trainer configuration successfully saved to ./hallucination_workspace_ohu_neg_v3_jun16/xgboost_model/config.json


In [7]:
print("\n--- Training Stage Complete ---")
print(f"Final Test Partition Scores: {test_performance}")
print(f"Model Directory: {trainer.model_dir}\n")


--- Training Stage Complete ---
Final Test Partition Scores: {'ROC_AUC': 0.9341091856382375, 'PRC_AUC': 0.9237332741168237, 'threshold_accuracy': 0.8894736842105263, 'threshold_precision': 0.9054054054054054, 'threshold_recall': 0.8271604938271605}
Model Directory: ./hallucination_workspace_ohu_neg_v3_jun16/xgboost_model



In [9]:
# -----------------------------------------------------------------
# STAGE 2: THE INFRASTRUCTURE INFERENCE PHASE (Checker)
# -----------------------------------------------------------------
print("====================================================")
print("STAGE 2: DEPLOYING LIVE INFERENCE CHECKER Pipeline")
print("====================================================")

production_model_path = os.path.join(WORKSPACE_DIR, "xgboost_model")

ml_checker = HallucinationChecker(
    checker_type="ml",
    model_path=production_model_path
)

# Validate a completely new out-of-distribution pathway step
product = "O=C1[C@@]2(CC(C)C)N([C@]3([H])C(N2)=O)[C@](CC(C)C)(C(O[C@H]3C)=O)N1"
proposed_reactants = ["O=C(Cl)[C@@]1(CC(C)C)N([C@]2([H])C(=O)N1)[C@](CC(C)C)(C(O[C@H]2C)=O)N"]

# Run the live assessment loop
status, valid_pathways = ml_checker(product, proposed_reactants)
print(status, valid_pathways)


STAGE 2: DEPLOYING LIVE INFERENCE CHECKER Pipeline
Restoring saved XGBOOST pipeline from disk...
Loaded Decision Threshold: 0.5091
200 []


In [11]:
heur_checker = HallucinationChecker(checker_type="heuristic")
heur_pred = heur_checker("C1=CC=CC=C1", ["NNN", "P"])
print(f"Heuristic code prediction: {heur_pred}")



Heuristic code prediction: (400, [])
